In [1]:
import os
import pandas as pd
from pathlib import Path
import re
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# import librosa
import numpy as np
import matplotlib.pyplot as plt
# from transformers import Wav2Vec2Model, Wav2Vec2Processor
import IPython.display as ipd
import soundfile as sf
from tqdm import tqdm

import subprocess

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.svm import SVC

from scipy.stats import ttest_rel, ttest_1samp, ttest_ind
from scipy import stats

torch.manual_seed(42)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
interview_task_path = "Androids-Corpus/Androids-Corpus/Interview-Task/audio"
interview_data = []

for condition in os.listdir(interview_task_path):
    if condition == ".DS_Store":
        continue
    for clip in os.listdir(interview_task_path + "/" + condition):
        stem = clip.replace(".wav", "")
        uid, mid, t = stem.split("_");
        X = mid[0]
        G = mid[1]
        mm = int(mid[2:])
        interview_data.append({
            "speaker_id": f"{uid}_{X}",
            "condition": X,
            "gender": G,
            "age": mm,
            "education_level": t,
            "path": f"{interview_task_path}/{condition}/{clip}"
        })

interview_df = pd.DataFrame(interview_data)
print(interview_df.head())
print(interview_df[interview_df["condition"] == "P"]["path"].iloc[0])
print(interview_df.shape)

  speaker_id condition gender  age education_level  \
0       01_C         C      F   56               1   
1       02_C         C      M   57               2   
2       03_C         C      F   30               3   
3       04_C         C      F   57               3   
4       05_C         C      F   41               3   

                                                path  
0  Androids-Corpus/Androids-Corpus/Interview-Task...  
1  Androids-Corpus/Androids-Corpus/Interview-Task...  
2  Androids-Corpus/Androids-Corpus/Interview-Task...  
3  Androids-Corpus/Androids-Corpus/Interview-Task...  
4  Androids-Corpus/Androids-Corpus/Interview-Task...  
Androids-Corpus/Androids-Corpus/Interview-Task/audio/PT/01_PM58_2.wav
(116, 6)


In [4]:
interview_df["gender"].value_counts()
interview_df["condition"].value_counts()
interview_df["path"][0]

'Androids-Corpus/Androids-Corpus/Interview-Task/audio/HC/01_CF56_1.wav'

In [5]:
label_of = {sid: 1 if condition == "P" else 0 for sid, condition in zip(interview_df.speaker_id, interview_df.condition)}
gender_of = dict(zip(interview_df.speaker_id, interview_df.gender))
len(label_of.keys())

116

In [6]:
meta = []
for _, row in interview_df.iterrows():

    info = sf.info(row["path"])
    meta.append({
        "speaker_id":    row["speaker_id"],
        "samplerate": info.samplerate,
        "channels":   info.channels,
        "duration":   info.duration,
    })

meta_df = pd.DataFrame(meta)
meta_df.head(5)

,speaker_id,samplerate,channels,duration
0,01_C,44100,1,243.789297
1,02_C,44100,1,255.709297
2,03_C,44100,1,249.909297
3,04_C,44100,1,261.399297
4,05_C,44100,1,291.149297


In [ ]:
meta_df["condition"] = meta_df["speaker_id"].map(
    dict(zip(interview_df["speaker_id"], interview_df["condition"]))
)
meta_df["gender"] = meta_df["speaker_id"].map(gender_of)

print("Overall duration (seconds):")
print(meta_df["duration"].describe())

print("\nBy condition (P = depressed, C = control):")
print(meta_df.groupby("condition")["duration"].agg(["mean", "std", "count"]))

print("\nSamplerate check:")
print(meta_df["samplerate"].value_counts())

p_durations = meta_df[meta_df["condition"] == "P"]["duration"]
c_durations = meta_df[meta_df["condition"] == "C"]["duration"]

t_stat, p_val = ttest_ind(p_durations, c_durations)
print(f"t-test: t={t_stat:.2f}, p={p_val:.6f}")

Overall duration (seconds):
count    116.000000
mean     229.843124
std       86.593584
min       70.649297
25%      159.514297
50%      240.779297
75%      272.461797
max      579.372698
Name: duration, dtype: float64

By condition (P = depressed, C = control):
                 mean        std  count
condition                              
C          268.022374  45.293864     52
P          198.822483  99.227922     64

Samplerate check (should be constant if corpus is consistent):
samplerate
44100    116
Name: count, dtype: int64
t-test: t=-4.65, p=0.000009


In [8]:
SMILE = r"C:\Desertation\opensmile-3.0.2-windows-x86_64\opensmile-3.0.2-windows-x86_64\bin\SMILExtract.exe"

r = subprocess.run([SMILE, "-h"], capture_output=True, text=True)
print(r.stdout[:300])
print("STDERR:", r.stderr[:300])


STDERR:  
   openSMILE version 3.0.1 (Rev. unknown)
   Build date: 2023-10-19T14:13:37Z
   Build branch: 'unknown'
   (c) 2022 by audEERING GmbH
   All rights reserved. See the file COPYING for license terms.


In [9]:
CONF = r"C:\Desertation\Opensmile_dd.conf"

def extract(wav_path, out_csv):
    r = subprocess.run(
        [SMILE, "-C", CONF, "-I", str(wav_path), "-O", str(out_csv)],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"{wav_path}\n{r.stderr[-400:]}")
    return pd.read_csv(out_csv)

npz_path = "androids_is09_02.npz"

if not os.path.exists(npz_path):
    features = {}
    for _, row in interview_df.iterrows():
        sid = row["speaker_id"]
        out_csv = f"features/{sid}.csv"
        df = extract(row["path"], out_csv)
        features[sid] = df.values.astype(np.float32)

    np.savez_compressed(npz_path, **features)
    print("Feature extraction complete")
else:
    print("Feature extraction already completed")

Feature extraction already completed


In [10]:
# import soundfile as sf
# import pandas as pd

# path = "Androids-Corpus/Androids-Corpus/Interview-Task/audio/HC/01_CF56_1.wav"
# info = sf.info(path)
# duration = info.frames / info.samplerate
# print("Duration:", duration, "seconds")

# out_csv = "test_01.csv"
# extract(path, out_csv=out_csv)

# df = pd.read_csv(out_csv, sep=",")
# print("Rows (frames):", len(df))
# print("Expected frames at 33ms hop:", duration / 0.033)

In [11]:
data = np.load(npz_path)
features = {k: data[k] for k in data.files}


In [12]:
fold_list = pd.read_csv("Androids-Corpus/Androids-corpus/fold-lists.csv", header=None, skiprows=2)
print(fold_list.head(10))

interview_folds = {}
for i, col in enumerate(range(7, 12), start=1):
    speakers = fold_list[col].dropna().str.strip("'").tolist()
    interview_folds[i] = speakers

            0            1            2            3            4   5   6   \
0  '01_CF56_1'  '05_CF41_3'  '06_CF44_2'  '07_CF50_2'  '03_CF30_3' NaN NaN   
1  '02_CM57_2'  '11_CF44_2'  '15_CF53_3'  '10_CF51_2'  '04_CF57_3' NaN NaN   
2  '09_CF56_3'  '32_CF22_3'  '20_CM51_3'  '12_CF36_1'  '08_CF42_2' NaN NaN   
3  '21_CF58_3'  '46_CF39_3'  '23_CF55_3'  '14_CF49_3'  '13_CF45_2' NaN NaN   
4  '22_CF50_3'  '47_CF61_2'  '26_CM31_3'  '16_CF33_4'  '17_CF55_3' NaN NaN   
5  '25_CF59_3'  '07_PM39_4'  '27_CF63_4'  '28_CF34_3'  '18_CM64_3' NaN NaN   
6  '33_CF46_3'  '13_PF58_2'  '30_CF62_3'  '29_CF34_3'  '19_CF62_4' NaN NaN   
7  '41_CM71_2'  '14_PF35_3'  '31_CF55_2'  '36_CF59_2'  '24_CM63_3' NaN NaN   
8  '55_CM29_3'  '15_PM63_4'  '37_CF69_1'  '42_CF53_4'  '38_CM27_3' NaN NaN   
9  '56_CM23_3'  '21_PM33_3'  '39_CM27_3'  '48_CF34_3'  '40_CF59_1' NaN NaN   

            7            8            9            10           11  
0  '01_CF56_1'  '12_CF36_1'  '03_CF30_3'  '05_CF41_3'  '06_CF44_2'  
1  

In [13]:
fold_summary = {}

for fold in interview_folds:
    if fold not in fold_summary:
        fold_summary[fold] = {'P': 0, 'C': 0}
    for speaker in interview_folds[fold]:
        speaker_type = speaker.split("_")[1][0]
        fold_summary[fold][speaker_type] += 1

print(f"Class ballance summary in each fold\n{fold_summary}")

Class ballance summary in each fold
{1: {'P': 12, 'C': 12}, 2: {'P': 13, 'C': 10}, 3: {'P': 10, 'C': 13}, 4: {'P': 18, 'C': 5}, 5: {'P': 11, 'C': 12}}


In [14]:
train_speakers = [s[:4] for f, sp in interview_folds.items() if f != 1 for s in sp]
X_train = np.vstack([data[s] for s in train_speakers])
print(X_train.shape)
total = sum(len(data[s]) for s in data)
print(total)
print(X_train.dtype, X_train.nbytes / 1e9, "GB")

(646710, 32)
808154
float32 0.08277888 GB
